In [35]:
%load_ext autoreload
%autoreload 2

from scalargrad.nn import MLP
from scalargrad.value import Value
n = MLP(3, [4, 4, 1], output_activation=Value.tanh)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
len(n.parameters())

41

In [37]:
xs: list[list[Value | float]] = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0], 
]

ys = [[1.0], [-1.0], [-1.0], [1.0]]
ypred = [n(x) for x in xs]
ypred

[[Value(data=0.24112061140460225, grad=0.0) , op=tanh],
 [Value(data=0.22122997907140834, grad=0.0) , op=tanh],
 [Value(data=0.1975401881557351, grad=0.0) , op=tanh],
 [Value(data=0.19435253021735474, grad=0.0) , op=tanh]]

In [ ]:
k = 1

loss = sum(((yout[0] - ygt[0])**2 for ygt, yout in zip(ys, ypred)), Value(0.0))
while(loss.data > 0.00001 and k < 20000):
    k += 1
    ypred = [n(x) for x in xs]
    loss = sum(((yout[0] - ygt[0])**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

    n.zero_grad()
    loss.backward()

    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, f"{loss.data:.8f}")

In [39]:
ypred = [n(x) for x in xs]
ypred

[[Value(data=0.9999804628380004, grad=0.0) , op=tanh],
 [Value(data=-0.9967799744035316, grad=0.0) , op=tanh],
 [Value(data=-0.9941932119468743, grad=0.0) , op=tanh],
 [Value(data=0.9963569727661732, grad=0.0) , op=tanh]]

In [43]:
acts = []
for x in xs:
    h = x
    per_layer = []
    for layer in n.layers:
        h = layer(h)
        per_layer.append([v.data for v in h])
    acts.append(per_layer)

for li in range(len(n.layers) - 1):
    for ni in range(len(n.layers[li].neurons)):
        values = [acts[s][li][ni] for s in range(len(xs))]
        dead = all(v == 0 for v in values)
        print(f"layer {li} neuron {ni}: {[round(v, 3) for v in values]}"
              f"{'  ← DEAD' if dead else ''}")

layer 0 neuron 0: [0, 0, 0, 0]  ← DEAD
layer 0 neuron 1: [0, 0.492, 0, 0]
layer 0 neuron 2: [4.281, 0, 0, 2.559]
layer 0 neuron 3: [2.485, 1.518, 1.109, 0.98]
layer 1 neuron 0: [1.709, 1.074, 0.911, 0.866]
layer 1 neuron 1: [3.278, 0.155, 0.008, 1.829]
layer 1 neuron 2: [0, 1.86, 1.568, 0.0]
layer 1 neuron 3: [5.061, 0, 0, 3.104]
